# Gold Macro Indicators

Heavy notebook entrypoint for macro-domain Gold indicators.

Supported branches:

- `source_system=ecb`: maps Silver ECB FX rates directly into `gld_macro.dp_macro_indicators`
- `source_system=fred`: collapses Silver FRED revisions to the latest available revision per `(series_id, observation_date)` and maps them into the same Gold table

This notebook writes only to `market_macro.gld_macro.dp_macro_indicators`.

FRED metadata is mandatory at Gold time. If requested series are missing metadata, units, or frequency, the notebook fails fast.

In [ ]:
import uuid
from datetime import datetime, timedelta, timezone

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

UTC = timezone.utc
FRED_INDICATOR_GROUPS = {
    "CPIAUCSL": "inflation",
    "FEDFUNDS": "policy_rate",
    "GDP": "output",
}


def parse_iso_date(field_name: str, raw_value: str):
    try:
        return datetime.strptime(raw_value, "%Y-%m-%d").date()
    except ValueError as exc:
        raise ValueError(f"{field_name} must be in YYYY-MM-DD format") from exc


def parse_quote_currencies(raw_value: str) -> list[str]:
    quote_currencies = [item.strip().upper() for item in raw_value.split(",") if item.strip()]
    quote_currencies = list(dict.fromkeys(quote_currencies))
    if not quote_currencies:
        raise ValueError("quote_currencies cannot be empty")

    for currency in quote_currencies:
        if len(currency) != 3 or not currency.isalpha():
            raise ValueError(f"quote_currencies must contain 3-letter ISO codes, got: {currency}")

    return quote_currencies


def parse_series_ids(raw_value: str) -> list[str]:
    series_ids = [item.strip().upper() for item in raw_value.split(",") if item.strip()]
    series_ids = list(dict.fromkeys(series_ids))
    if not series_ids:
        raise ValueError("series_ids cannot be empty")
    return series_ids


def resolve_date_window(mode: str, start_date_raw: str, end_date_raw: str, lookback_days_raw: str):
    latest_complete_date = datetime.now(UTC).date() - timedelta(days=1)
    normalized_mode = mode.strip().lower()

    if normalized_mode not in {"backfill", "incremental"}:
        raise ValueError("mode must be either 'backfill' or 'incremental'")

    if normalized_mode == "backfill":
        if not start_date_raw or not end_date_raw:
            raise ValueError("backfill mode requires both start_date and end_date")
        start_date = parse_iso_date("start_date", start_date_raw)
        end_date = parse_iso_date("end_date", end_date_raw)
    else:
        try:
            lookback_days = int(lookback_days_raw or "0")
        except ValueError as exc:
            raise ValueError("lookback_days must be an integer in incremental mode") from exc

        if lookback_days < 1:
            raise ValueError("lookback_days must be >= 1 in incremental mode")

        end_date = parse_iso_date("end_date", end_date_raw) if end_date_raw else latest_complete_date
        start_date = end_date - timedelta(days=lookback_days - 1)

    if start_date > end_date:
        raise ValueError("start_date cannot be after end_date")

    if end_date > latest_complete_date:
        raise ValueError(
            f"end_date must be <= latest completed UTC day: {latest_complete_date.isoformat()}"
        )

    return start_date, end_date


def ensure_table_exists(table_name: str):
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Required table {table_name} does not exist. Run 00_platform_setup_catalog_schema.ipynb first."
        )


def collect_counts(df, key_column: str, count_alias: str) -> dict[str, int]:
    counts = (
        df.groupBy(key_column)
        .count()
        .withColumnRenamed("count", count_alias)
        .collect()
    )
    return {row[key_column]: int(row[count_alias]) for row in counts}


def create_indicator_group_expr():
    mapping_items = []
    for series_id, indicator_group in FRED_INDICATOR_GROUPS.items():
        mapping_items.extend([F.lit(series_id), F.lit(indicator_group)])
    mapping_expr = F.create_map(mapping_items)
    return F.element_at(mapping_expr, F.col("series_id"))


spark.conf.set("spark.sql.session.timeZone", "UTC")

dbutils.widgets.text("catalog", "market_macro")
dbutils.widgets.text("source_system", "ecb")
dbutils.widgets.text("quote_currencies", "USD")
dbutils.widgets.text("series_ids", "CPIAUCSL,FEDFUNDS,GDP")
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "backfill"])
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("lookback_days", "")
dbutils.widgets.text("run_id", str(uuid.uuid4()))


In [ ]:
catalog = dbutils.widgets.get("catalog").strip() or "market_macro"
source_system = dbutils.widgets.get("source_system").strip().lower()
mode = dbutils.widgets.get("mode").strip().lower()
run_id = dbutils.widgets.get("run_id").strip()
raw_lookback_days = dbutils.widgets.get("lookback_days").strip()
lookback_days_raw = raw_lookback_days or ("90" if source_system == "fred" else "7")
start_date, end_date = resolve_date_window(
    mode=mode,
    start_date_raw=dbutils.widgets.get("start_date").strip(),
    end_date_raw=dbutils.widgets.get("end_date").strip(),
    lookback_days_raw=lookback_days_raw,
)
computed_at = datetime.now(UTC)

target_table = f"{catalog}.gld_macro.dp_macro_indicators"
ensure_table_exists(target_table)

if source_system == "ecb":
    source_table = f"{catalog}.slv_macro.ecb_fx_ref_rates_daily"
    ensure_table_exists(source_table)
    quote_currencies = parse_quote_currencies(dbutils.widgets.get("quote_currencies").strip())

    silver_df = (
        spark.table(source_table)
        .select(
            F.upper(F.col("base_currency")).alias("base_currency"),
            F.upper(F.col("quote_currency")).alias("quote_currency"),
            "rate_date",
            "rate",
        )
        .filter(F.col("quote_currency").isin(quote_currencies))
        .filter((F.col("rate_date") >= F.lit(start_date)) & (F.col("rate_date") <= F.lit(end_date)))
    )

    rows_read = silver_df.count()
    if rows_read == 0:
        result = {
            "status": "success_empty",
            "source_system": source_system,
            "mode": mode,
            "start_date": start_date.isoformat(),
            "end_date": end_date.isoformat(),
            "quote_currencies": quote_currencies,
            "source_table": source_table,
            "target_table": target_table,
            "rows_read": 0,
            "rows_ready": 0,
            "rows_to_update": 0,
            "rows_to_insert": 0,
            "rows_merged": 0,
            "run_id": run_id,
            "per_indicator_rows_ready": {},
        }
    else:
        gold_df = (
            silver_df
            .withColumn(
                "indicator_id",
                F.concat_ws("_", F.lit("ECB"), F.lit("FX"), F.lit("REF"), F.col("base_currency"), F.col("quote_currency")),
            )
            .withColumnRenamed("rate_date", "observation_date")
            .withColumn("source_system", F.lit("ecb"))
            .withColumn("indicator_group", F.lit("fx_ref_rate"))
            .withColumn("value", F.col("rate"))
            .withColumn("unit", F.concat(F.col("quote_currency"), F.lit(" per "), F.col("base_currency")))
            .withColumn("frequency", F.lit("D"))
            .withColumn("is_official", F.lit(True))
            .withColumn("derivation_method", F.lit("official_source"))
            .withColumn("derived_from_indicator_id", F.lit(None).cast("string"))
            .withColumn("computed_at", F.lit(computed_at))
            .withColumn("run_id", F.lit(run_id))
            .select(
                "indicator_id",
                "observation_date",
                "source_system",
                "indicator_group",
                "value",
                "unit",
                "frequency",
                "is_official",
                "derivation_method",
                "derived_from_indicator_id",
                "computed_at",
                "run_id",
            )
        )

        duplicate_key_count = (
            gold_df.groupBy("indicator_id", "observation_date")
            .count()
            .filter(F.col("count") > 1)
            .count()
        )
        if duplicate_key_count > 0:
            raise RuntimeError(
                f"Detected {duplicate_key_count} duplicate Gold ECB indicator keys after mapping."
            )

        rows_ready = gold_df.count()
        per_indicator_rows_ready = collect_counts(gold_df, "indicator_id", "rows_ready")
        existing_key_count = (
            gold_df.select("indicator_id", "observation_date")
            .join(
                spark.table(target_table).select("indicator_id", "observation_date"),
                on=["indicator_id", "observation_date"],
                how="inner",
            )
            .count()
        )

        DeltaTable.forName(spark, target_table).alias("tgt").merge(
            gold_df.alias("src"),
            "tgt.indicator_id = src.indicator_id AND tgt.observation_date = src.observation_date",
        ).whenMatchedUpdate(
            set={
                "source_system": "src.source_system",
                "indicator_group": "src.indicator_group",
                "value": "src.value",
                "unit": "src.unit",
                "frequency": "src.frequency",
                "is_official": "src.is_official",
                "derivation_method": "src.derivation_method",
                "derived_from_indicator_id": "src.derived_from_indicator_id",
                "computed_at": "src.computed_at",
                "run_id": "src.run_id",
            }
        ).whenNotMatchedInsertAll().execute()

        display(gold_df.orderBy("indicator_id", "observation_date"))
        result = {
            "status": "success",
            "source_system": source_system,
            "mode": mode,
            "start_date": start_date.isoformat(),
            "end_date": end_date.isoformat(),
            "quote_currencies": quote_currencies,
            "source_table": source_table,
            "target_table": target_table,
            "rows_read": rows_read,
            "rows_ready": rows_ready,
            "rows_to_update": existing_key_count,
            "rows_to_insert": rows_ready - existing_key_count,
            "rows_merged": rows_ready,
            "run_id": run_id,
            "per_indicator_rows_ready": per_indicator_rows_ready,
        }
elif source_system == "fred":
    source_table = f"{catalog}.slv_macro.fred_series_clean"
    metadata_table = f"{catalog}.brz_macro.raw_fred_series_metadata"
    ensure_table_exists(source_table)
    ensure_table_exists(metadata_table)
    series_ids = parse_series_ids(dbutils.widgets.get("series_ids").strip())
    revision_policy = "latest_available"

    silver_df = (
        spark.table(source_table)
        .select(
            "series_id",
            "observation_date",
            "realtime_start",
            "realtime_end",
            "value",
            "ingested_at",
        )
        .filter(F.col("series_id").isin(series_ids))
        .filter((F.col("observation_date") >= F.lit(start_date)) & (F.col("observation_date") <= F.lit(end_date)))
    )

    rows_read = silver_df.count()
    if rows_read == 0:
        result = {
            "status": "success_empty",
            "source_system": source_system,
            "mode": mode,
            "series_ids": series_ids,
            "revision_policy": revision_policy,
            "start_date": start_date.isoformat(),
            "end_date": end_date.isoformat(),
            "source_table": source_table,
            "metadata_table": metadata_table,
            "target_table": target_table,
            "rows_read": 0,
            "rows_after_revision_collapse": 0,
            "rows_ready": 0,
            "rows_to_update": 0,
            "rows_to_insert": 0,
            "rows_merged": 0,
            "run_id": run_id,
            "per_indicator_rows_ready": {},
        }
    else:
        metadata_window = Window.partitionBy("series_id").orderBy(
            F.col("ingested_at").desc(),
            F.col("payload_hash").desc(),
        )
        metadata_latest_df = (
            spark.table(metadata_table)
            .select("series_id", "units", "frequency_short", "ingested_at", "payload_hash")
            .filter(F.col("series_id").isin(series_ids))
            .withColumn("_row_number", F.row_number().over(metadata_window))
            .filter(F.col("_row_number") == 1)
            .drop("_row_number")
        )

        metadata_ids = {row["series_id"] for row in metadata_latest_df.select("series_id").collect()}
        missing_metadata_ids = sorted(set(series_ids) - metadata_ids)
        if missing_metadata_ids:
            raise RuntimeError(
                "Missing FRED metadata for requested series_ids: " + ", ".join(missing_metadata_ids)
            )

        metadata_missing_units = [
            row["series_id"]
            for row in metadata_latest_df.filter(F.col("units").isNull() | (F.trim(F.col("units")) == "")).select("series_id").collect()
        ]
        if metadata_missing_units:
            raise RuntimeError(
                "Missing FRED units for series_ids: " + ", ".join(sorted(metadata_missing_units))
            )

        metadata_missing_frequency = [
            row["series_id"]
            for row in metadata_latest_df.filter(
                F.col("frequency_short").isNull() | (F.trim(F.col("frequency_short")) == "")
            ).select("series_id").collect()
        ]
        if metadata_missing_frequency:
            raise RuntimeError(
                "Missing FRED frequency_short for series_ids: " + ", ".join(sorted(metadata_missing_frequency))
            )

        revision_window = Window.partitionBy("series_id", "observation_date").orderBy(
            F.col("realtime_start").desc(),
            F.col("realtime_end").desc(),
            F.col("ingested_at").desc(),
        )

        latest_revision_df = (
            silver_df
            .withColumn("_row_number", F.row_number().over(revision_window))
            .filter(F.col("_row_number") == 1)
            .drop("_row_number")
        )
        rows_after_revision_collapse = latest_revision_df.count()

        gold_df = (
            latest_revision_df
            .join(metadata_latest_df.select("series_id", "units", "frequency_short"), on="series_id", how="inner")
            .withColumn("indicator_id", F.concat(F.lit("FRED_"), F.col("series_id")))
            .withColumn("source_system", F.lit("fred"))
            .withColumn("indicator_group", create_indicator_group_expr())
            .withColumn("unit", F.col("units"))
            .withColumn("frequency", F.col("frequency_short"))
            .withColumn("is_official", F.lit(True))
            .withColumn("derivation_method", F.lit("official_source"))
            .withColumn("derived_from_indicator_id", F.lit(None).cast("string"))
            .withColumn("computed_at", F.lit(computed_at))
            .withColumn("run_id", F.lit(run_id))
            .select(
                "indicator_id",
                "observation_date",
                "source_system",
                "indicator_group",
                "value",
                "unit",
                "frequency",
                "is_official",
                "derivation_method",
                "derived_from_indicator_id",
                "computed_at",
                "run_id",
            )
        )

        duplicate_key_count = (
            gold_df.groupBy("indicator_id", "observation_date")
            .count()
            .filter(F.col("count") > 1)
            .count()
        )
        if duplicate_key_count > 0:
            raise RuntimeError(
                f"Detected {duplicate_key_count} duplicate Gold FRED indicator keys after revision collapse."
            )

        rows_ready = gold_df.count()
        per_indicator_rows_ready = collect_counts(gold_df, "indicator_id", "rows_ready")
        existing_key_count = (
            gold_df.select("indicator_id", "observation_date")
            .join(
                spark.table(target_table).select("indicator_id", "observation_date"),
                on=["indicator_id", "observation_date"],
                how="inner",
            )
            .count()
        )

        DeltaTable.forName(spark, target_table).alias("tgt").merge(
            gold_df.alias("src"),
            "tgt.indicator_id = src.indicator_id AND tgt.observation_date = src.observation_date",
        ).whenMatchedUpdate(
            set={
                "source_system": "src.source_system",
                "indicator_group": "src.indicator_group",
                "value": "src.value",
                "unit": "src.unit",
                "frequency": "src.frequency",
                "is_official": "src.is_official",
                "derivation_method": "src.derivation_method",
                "derived_from_indicator_id": "src.derived_from_indicator_id",
                "computed_at": "src.computed_at",
                "run_id": "src.run_id",
            }
        ).whenNotMatchedInsertAll().execute()

        display(gold_df.orderBy("indicator_id", "observation_date"))
        result = {
            "status": "success",
            "source_system": source_system,
            "mode": mode,
            "series_ids": series_ids,
            "revision_policy": revision_policy,
            "start_date": start_date.isoformat(),
            "end_date": end_date.isoformat(),
            "source_table": source_table,
            "metadata_table": metadata_table,
            "target_table": target_table,
            "rows_read": rows_read,
            "rows_after_revision_collapse": rows_after_revision_collapse,
            "rows_ready": rows_ready,
            "rows_to_update": existing_key_count,
            "rows_to_insert": rows_ready - existing_key_count,
            "rows_merged": rows_ready,
            "run_id": run_id,
            "per_indicator_rows_ready": per_indicator_rows_ready,
        }
else:
    raise ValueError("source_system must be one of: ecb, fred")

result
